# 1. Install required packages

Installs and upgrades LangChain core, Google Gemini GenAI integration, OpenTelemetry SDK, and OpenTelemetry stdout exporter for building observable LLM applications.

In [ ]:
!pip install -qU langchain-core langchain-google-genai opentelemetry-sdk opentelemetry-exporter-stdout

# Imports

Imports modules for environment setup, Google Gemini LLM integration, and OpenTelemetry tracing configuration in a Google Colab environment.

In [ ]:
import os
import json
import time
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.resources import Resource

# 2. Configure OpenTelemetry tracing

Configures OpenTelemetry tracing with a custom service resource, console span exporter, and dedicated tracer for Gemini agent observability.

Key Components:
* Resource Definition: service.name="gemini-observability" for trace categorization

* Tracer Provider: Initialized with the custom resource

* Console Exporter: SimpleSpanProcessor(ConsoleSpanExporter()) for real-time trace output

* Tracer Creation: Named "gemini.agent" for specific instrumentation

In [ ]:
resource = Resource(attributes={"service.name": "gemini-observability"})
trace.set_tracer_provider(TracerProvider(resource=resource))
span_processor = SimpleSpanProcessor(ConsoleSpanExporter())  # Basic console exporter
trace.get_tracer_provider().add_span_processor(span_processor)
tracer = trace.get_tracer("gemini.agent")

# 3. Observable function with Gemini

Defines a function that performs Gemini 1.5 Flash inference with OpenTelemetry tracing, logs structured query/response metadata including latency, and returns the LLM's response content.

Key Features:
* Tracing: Uses "gemini4_inference" span with custom attributes (model, question)

* Secure Initialization: Dynamically sets Gemini API key from Colab secrets

* Logging: Outputs structured JSON with timestamp, question, response, latency, and model

* Response Handling: Returns raw LLM response content for downstream use

This implementation combines observability (tracing + logging) with Gemini 1.5 Flash inference in a single reusable function.

In [ ]:
def traced_chain(question: str) -> str:
    with tracer.start_as_current_span("gemini4_inference") as span:
        start_time = time.time()

        # Set API key and initialize Gemini 4
        os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_NEW")
        llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest", temperature=0.7)

        # Add custom attributes to span
        span.set_attribute("model", "gemini-1.5-flash-latest")
        span.set_attribute("question", question)

        # Get response
        response = llm.invoke(question)
        latency = time.time() - start_time

        # Log details
        log_entry = {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "question": question,
            "response": response.content,
            "latency": f"{latency:.2f}s",
            "model": "gemini-1.5-flash-latest"
        }
        print(json.dumps(log_entry, indent=2))
        return response.content

# 4. Run with tracing

Executes an observable Gemini 1.5 Flash inference to explain quantum entanglement in simple terms, with error-handled response formatting and emoji-enhanced output.

Key Actions:
* LLM Invocation: Calls traced_chain() with a physics explanation request

* Output Formatting: Prints response with emoji prefix for visual distinction

* Error Handling: Catches and displays -prefixed errors if inference fails

* Observability Integration: Leverages the full tracing/logging from traced_chain()

In [ ]:
try:
    response = traced_chain("Explain quantum entanglement in simple terms")
    print(f"\n💎 Final Response: {response}")
except Exception as e:
    print(f"❌ Error: {str(e)}")

ERROR: Could not find a version that satisfies the requirement opentelemetry-exporter-stdout (from versions: none)
ERROR: No matching distribution found for opentelemetry-exporter-stdout


{
  "timestamp": "2025-06-19 11:48:55",
  "question": "Explain quantum entanglement in simple terms",
  "response": "Imagine you have two coins, magically linked.  You put one in a box and send it to a friend far away.  When you open your box and see \"heads,\" you *instantly* know your friend's coin is \"tails,\" even before they open their box.  That's kind of like entanglement.\n\nIn quantum physics, two particles can be linked in a special way.  Their properties (like spin, which is like a tiny internal rotation) are correlated.  No matter how far apart they are, measuring the property of one particle *instantly* tells you the property of the other, even faster than light could travel between them.\n\nIt's important to note:\n\n* **It's not about sending information faster than light.** You can't use entanglement to send messages.  You only know the other particle's state *after* you've measured yours.\n* **It's weird.**  Even physicists don't fully understand how it works, but it'